"""
 ## GEOAI PADDY vs NOT-PADDY CLASSIFICATION PIPELINE
 - Sensor     : PlanetScope (4-band: B, G, R, NIR — 3m resolution)
 - Season     : Rabi 2025-26 | District: Koraput | Block: Borigumma
 - Author     : Senior GeoAI / Remote Sensing Engineer
 - Strategy   : Single-image maximum feature extraction → ensemble ML

PIPELINE OVERVIEW
─────────────────
 1. Read PlanetScope GeoTIFF + UDM2-style nodata masking
 2. Compute 10 vegetation indices
 3. Extract GLCM texture features (6 metrics × multi-band)
 4. Compute local spatial statistics (mean, variance, std) — 3×3 & 5×5
 5. Edge / structure features (Sobel, Laplacian, morphological gradient)
 6. PCA components + band ratios + entropy filters
 7. Stack → flat feature matrix → align with training shapefile samples
 8. Train Random Forest + XGBoost with SMOTE balancing + Optuna HPO
 9. Probability threshold optimisation (F1-optimal)
10. Classify full raster → export classified GeoTIFF + probability raster
11. Feature importance plot + confusion matrix + accuracy report
"""

In [2]:
pip install skimage

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [3 lines of output]
      
      *** Please install the `scikit-image` package (instead of `skimage`) ***
      
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'skimage' when getting requirements to build wheel


In [5]:


# ─── Standard library ────────────────────────────────────────────────────────
import os
import warnings
import time
from pathlib import Path

warnings.filterwarnings("ignore")

# ─── Core scientific stack ────────────────────────────────────────────────────
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

# ─── Geospatial stack ─────────────────────────────────────────────────────────
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.enums import Resampling
import geopandas as gpd
from shapely.geometry import mapping

# ─── Image processing ─────────────────────────────────────────────────────────
# ─── Image processing ─────────────────────────────────────────────────────────
from skimage.feature import graycomatrix, graycoprops

# Basic filters
from skimage.filters import sobel, laplace

# Entropy filter (correct import)
from skimage.filters.rank import entropy as skimage_entropy

# Morphology
from skimage.morphology import disk

# Utilities
from skimage.util import img_as_ubyte

# ─── Scipy image processing ───────────────────────────────────────────────────
from scipy.ndimage import (
    generic_filter,
    uniform_filter,
    sobel as scipy_sobel,
    laplace as scipy_laplace,
    grey_dilation,
    grey_erosion
)

from scipy.signal import convolve2d

# ─── ML stack ─────────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─── Plotting ─────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap

print("✅ All imports successful")

# ═══════════════════════════════════════════════════════════════════════════════
#  0. CONFIGURATION — edit paths here only
# ═══════════════════════════════════════════════════════════════════════════════

CFG = {
    # ── Inputs ──────────────────────────────────────────────────────────────
    "raster_path"  : r"\\192.168.0.243\paddy_ai\Planet_Image_Rabi_2526\Koraput\Borigumma_Rabi_2526.tif",
    "shp_path"     : r"E:\Rabi_Paddy_2526\LOT1\Koraput\TS\Borigumma_TS_TS.shp",
    "class_field"  : "class",          # 0 = Not Paddy | 1 = Paddy

    # ── Band order (1-indexed, PlanetScope standard) ─────────────────────────
    "band_blue"    : 1,
    "band_green"   : 2,
    "band_red"     : 3,
    "band_nir"     : 4,

    # ── Outputs ──────────────────────────────────────────────────────────────
    "out_dir"      : r"E:\Rabi_Paddy_2526\LOT1\Koraput\Output",

    # ── Feature engineering ──────────────────────────────────────────────────
    "glcm_distances": [1, 2],           # GLCM pixel distances
    "glcm_angles"   : [0, np.pi/4, np.pi/2, 3*np.pi/4],
    "window_sizes"  : [3, 5],           # local statistics windows
    "n_pca_components": 4,

    # ── Model ────────────────────────────────────────────────────────────────
    "n_optuna_trials": 30,             # increase for better HPO
    "cv_folds"      : 5,
    "random_state"  : 42,
    "nodata_val"    : 0,               # PlanetScope nodata
    "scale_factor"  : 10000.0,        # DN → reflectance divisor
}

# Create output directory
os.makedirs(CFG["out_dir"], exist_ok=True)
out = Path(CFG["out_dir"])

print(f"📁 Output directory: {out}")


# ═══════════════════════════════════════════════════════════════════════════════
#  1. READ RASTER
# ═══════════════════════════════════════════════════════════════════════════════

def read_raster(path: str, scale: float = 10000.0, nodata: int = 0):
    """
    Read PlanetScope GeoTIFF.

    Returns
    -------
    bands   : dict  {name: 2D float32 array}  — reflectance [0..1]
    meta    : rasterio meta dict
    nodata_mask : bool array True where valid
    """
    print("\n📡 Reading raster …")
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)       # (4, H, W)
        meta = src.meta.copy()
        nd   = src.nodata if src.nodata is not None else nodata

    # Build validity mask: True = valid pixel
    nodata_mask = np.all(data != nd, axis=0)

    # Scale to reflectance; clip to [0, 1]
    data = np.where(nodata_mask[np.newaxis], data / scale, np.nan)
    data = np.clip(data, 0.0, 1.0)

    H, W = data.shape[1], data.shape[2]
    print(f"   Shape: {H} × {W}  |  Valid pixels: {nodata_mask.sum():,}")

    bands = {
        "Blue": data[0], "Green": data[1],
        "Red" : data[2], "NIR"  : data[3],
    }
    return bands, meta, nodata_mask


# ═══════════════════════════════════════════════════════════════════════════════
#  2. VEGETATION INDICES
# ═══════════════════════════════════════════════════════════════════════════════

def safe_divide(a, b, fill=0.0):
    """Division with zero-denominator guard."""
    with np.errstate(invalid="ignore", divide="ignore"):
        out = np.where(np.abs(b) > 1e-6, a / b, fill)
    return out.astype(np.float32)


def compute_indices(B, G, R, N):
    """
    Compute 10 vegetation / water indices.

    PlanetScope band mapping:
        B=Blue  G=Green  R=Red  N=NIR
    """
    print("🌿 Computing vegetation indices …")
    eps = 1e-6
    L   = 0.5     # SAVI soil-brightness correction factor

    idx = {}

    # ── Core indices ──────────────────────────────────────────────────────────
    idx["NDVI"]  = safe_divide(N - R, N + R)                         # Paddy phenology tracker
    idx["GNDVI"] = safe_divide(N - G, N + G)                         # Canopy chlorophyll
    idx["SAVI"]  = safe_divide(1.5 * (N - R), N + R + L)             # Soil-adjusted
    idx["MSAVI2"]= (2*N + 1 - np.sqrt((2*N+1)**2 - 8*(N-R))) / 2     # Modified SAVI

    # ── EVI (requires Blue to reduce atmospheric + canopy bg) ─────────────────
    idx["EVI"]   = safe_divide(
        2.5 * (N - R),
        N + 6*R - 7.5*B + 1
    )

    # ── Water / moisture ──────────────────────────────────────────────────────
    idx["NDWI"]  = safe_divide(G - N, G + N)                         # Open water / paddy flooding

    # ── Ratio-based ───────────────────────────────────────────────────────────
    idx["RVI"]   = safe_divide(N, R)                                   # Ratio Vegetation Index
    idx["DVI"]   = (N - R).astype(np.float32)                         # Difference VI
    idx["IPVI"]  = safe_divide(N, N + R)                               # Infrared Percentage VI
    idx["OSAVI"] = safe_divide(N - R, N + R + 0.16)                   # Optimised SAVI

    for k, v in idx.items():
        idx[k] = np.clip(v, -1.5, 1.5).astype(np.float32)

    print(f"   Generated {len(idx)} indices")
    return idx


# ═══════════════════════════════════════════════════════════════════════════════
#  3. GLCM TEXTURE FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

def _glcm_window(patch: np.ndarray, distances, angles, props) -> np.ndarray:
    """Compute averaged GLCM properties for a single 2D patch."""
    patch_u8 = img_as_ubyte(np.clip(patch, 0, 1))
    glcm     = graycomatrix(patch_u8, distances, angles,
                             levels=256, symmetric=True, normed=True)
    return np.array([graycoprops(glcm, p).mean() for p in props],
                    dtype=np.float32)


def compute_glcm_texture(band2d: np.ndarray,
                         win: int = 9,
                         distances=(1, 2),
                         angles=(0, np.pi/4, np.pi/2, 3*np.pi/4),
                         band_name: str = "band") -> dict:
    """
    Sliding-window GLCM texture extraction.

    Computes: contrast, dissimilarity, homogeneity, ASM, energy, correlation.

    win : int  — window side length (odd preferred)
    """
    props   = ["contrast", "dissimilarity", "homogeneity",
               "ASM",      "energy",        "correlation"]
    pad     = win // 2
    H, W    = band2d.shape
    results = {f"GLCM_{band_name}_{p}": np.full((H, W), np.nan, np.float32)
               for p in props}

    band_pad = np.pad(band2d, pad, mode="reflect")

    print(f"   GLCM [{band_name}] {H}×{W} win={win}  … (may take a minute)")
    for i in range(H):
        for j in range(W):
            patch = band_pad[i:i+win, j:j+win]
            vals  = _glcm_window(patch, distances, angles, props)
            for k, p in enumerate(props):
                results[f"GLCM_{band_name}_{p}"][i, j] = vals[k]

    return results


def compute_all_glcm(bands: dict, win: int = 9,
                     distances=(1,), angles=(0, np.pi/2)) -> dict:
    """
    Compute GLCM on NIR and Red bands only for speed.
    NIR is most discriminative for vegetation texture.
    """
    print("\n🔲 Computing GLCM texture features …")
    out = {}
    for bname in ["NIR", "Red", "Green"]:
        out.update(compute_glcm_texture(
            bands[bname], win=win,
            distances=distances, angles=angles,
            band_name=bname
        ))
    print(f"   Generated {len(out)} GLCM feature maps")
    return out


# ═══════════════════════════════════════════════════════════════════════════════
#  4. LOCAL SPATIAL STATISTICS
# ═══════════════════════════════════════════════════════════════════════════════

def local_stats(band2d: np.ndarray, win: int, name: str) -> dict:
    """Local mean, variance, std using uniform filter."""
    mean_f = uniform_filter(band2d, size=win).astype(np.float32)
    mean_sq = uniform_filter(band2d**2, size=win).astype(np.float32)
    var_f  = np.clip(mean_sq - mean_f**2, 0, None).astype(np.float32)
    std_f  = np.sqrt(var_f).astype(np.float32)
    return {
        f"LocMean_{name}_w{win}": mean_f,
        f"LocVar_{name}_w{win}" : var_f,
        f"LocStd_{name}_w{win}" : std_f,
    }


def compute_local_stats(bands: dict, windows=(3, 5)) -> dict:
    print("\n📊 Computing local spatial statistics …")
    out = {}
    for win in windows:
        for bname in ["NIR", "Red", "Green", "NDVI"]:
            b = bands.get(bname)
            if b is not None:
                out.update(local_stats(b, win, bname))
    print(f"   Generated {len(out)} local stat maps")
    return out


# ═══════════════════════════════════════════════════════════════════════════════
#  5. EDGE / STRUCTURE FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

def compute_edge_features(bands: dict) -> dict:
    """Sobel, Laplacian, morphological gradient on NIR and NDVI."""
    print("\n⚡ Computing edge & structure features …")
    out = {}
    for bname in ["NIR", "Red"]:
        b   = np.nan_to_num(bands[bname], nan=0.0)
        b_u = np.clip(b, 0, 1).astype(np.float32)

        # Sobel magnitude
        sx  = scipy_sobel(b_u, axis=1)
        sy  = scipy_sobel(b_u, axis=0)
        out[f"Sobel_{bname}"]   = np.sqrt(sx**2 + sy**2).astype(np.float32)

        # Laplacian (2nd derivative → edge sharpness)
        out[f"Laplace_{bname}"] = np.abs(scipy_laplace(b_u)).astype(np.float32)

        # Morphological gradient = dilation − erosion
        struct = np.ones((3, 3))
        out[f"MorphGrad_{bname}"] = (
            grey_dilation(b_u, footprint=struct) -
            grey_erosion(b_u,  footprint=struct)
        ).astype(np.float32)

    print(f"   Generated {len(out)} edge/structure maps")
    return out


# ═══════════════════════════════════════════════════════════════════════════════
#  6. PCA + BAND RATIOS + ENTROPY
# ═══════════════════════════════════════════════════════════════════════════════

def compute_pca(bands: dict, n_components: int = 4,
                nodata_mask: np.ndarray = None) -> dict:
    """PCA across all 4 spectral bands."""
    print("\n🔬 Computing PCA components …")
    H, W = bands["Blue"].shape
    stack = np.stack([bands["Blue"], bands["Green"],
                      bands["Red"],  bands["NIR"]], axis=-1)   # (H,W,4)
    flat  = stack.reshape(-1, 4)

    if nodata_mask is not None:
        valid = nodata_mask.ravel()
        flat_valid = flat[valid]
    else:
        valid      = np.ones(H*W, bool)
        flat_valid = flat

    pca   = PCA(n_components=n_components, random_state=42)
    pcs   = pca.fit_transform(flat_valid)

    out_full = np.full((H*W, n_components), np.nan, np.float32)
    out_full[valid] = pcs.astype(np.float32)
    out_full = out_full.reshape(H, W, n_components)

    var_explained = pca.explained_variance_ratio_ * 100
    result = {}
    for i in range(n_components):
        result[f"PCA_{i+1}"] = out_full[:, :, i]
        print(f"   PC{i+1}: {var_explained[i]:.1f}% variance")

    return result


def compute_band_ratios(bands: dict) -> dict:
    """Targeted band ratios to amplify paddy vs plantation separation."""
    print("\n➗ Computing band ratios …")
    B, G, R, N = bands["Blue"], bands["Green"], bands["Red"], bands["NIR"]
    return {
        "Ratio_NIR_G"  : safe_divide(N, G),      # Elevated in paddy (dense canopy)
        "Ratio_NIR_B"  : safe_divide(N, B),
        "Ratio_R_G"    : safe_divide(R, G),      # Chlorophyll absorption
        "Ratio_NIR_R"  : safe_divide(N, R),      # Same as RVI — reinforced
        "Ratio_G_B"    : safe_divide(G, B),
        "Ratio_NRG"    : safe_divide(N - R, G),  # Normalised difference with green
    }


def compute_entropy_filter(bands: dict, radius: int = 3) -> dict:
    """Shannon entropy using skimage — captures textural randomness."""
    print("\n🌀 Computing entropy filter …")
    out = {}
    selem = disk(radius)
    for bname in ["NIR", "Red"]:
        b    = np.nan_to_num(bands[bname], nan=0.0)
        b_u8 = img_as_ubyte(np.clip(b, 0, 1))
        ent  = skimage_entropy(b_u8, selem).astype(np.float32)
        out[f"Entropy_{bname}_r{radius}"] = ent
    print(f"   Generated {len(out)} entropy maps")
    return out


# ═══════════════════════════════════════════════════════════════════════════════
#  7. FEATURE STACK
# ═══════════════════════════════════════════════════════════════════════════════

def build_feature_stack(bands, indices, glcm, local_st, edges,
                        pca_feats, ratios, entropy_feats):
    """
    Combine all feature maps into a single (H, W, F) array.
    Returns stack array and ordered feature names list.
    """
    print("\n🏗  Building feature stack …")
    all_feats = {}
    all_feats.update({k: bands[k] for k in ["Blue", "Green", "Red", "NIR"]})
    all_feats.update(indices)
    all_feats.update(glcm)
    all_feats.update(local_st)
    all_feats.update(edges)
    all_feats.update(pca_feats)
    all_feats.update(ratios)
    all_feats.update(entropy_feats)

    names = list(all_feats.keys())
    H, W  = bands["Blue"].shape
    stack = np.stack([all_feats[n] for n in names], axis=-1)   # (H, W, F)

    print(f"   Total features: {len(names)}")
    return stack, names


# ═══════════════════════════════════════════════════════════════════════════════
#  8. SAMPLE EXTRACTION FROM SHAPEFILE
# ═══════════════════════════════════════════════════════════════════════════════

def extract_samples(shp_path: str, raster_path: str,
                    feature_stack: np.ndarray,
                    feature_names: list,
                    class_field: str,
                    meta: dict):
    """
    Rasterise training polygons pixel-by-pixel using rasterio masking.

    Returns X (n_samples, n_features), y (n_samples,)
    """
    print("\n🗺  Extracting training samples from shapefile …")
    gdf = gpd.read_file(shp_path)
    print(f"   Training polygons: {len(gdf)}")
    print(f"   Class distribution:\n{gdf[class_field].value_counts()}")

    # Reproject shapefile to raster CRS if needed
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        transform  = src.transform

    if gdf.crs != raster_crs:
        print(f"   Reprojecting SHP from {gdf.crs} → {raster_crs}")
        gdf = gdf.to_crs(raster_crs)

    H, W, F = feature_stack.shape
    X_list, y_list = [], []

    for _, row in gdf.iterrows():
        geom  = [mapping(row.geometry)]
        label = int(row[class_field])

        # Create boolean pixel mask for this polygon
        from rasterio.features import geometry_mask
        poly_mask = ~geometry_mask(
            geom, transform=transform,
            invert=False, out_shape=(H, W)
        )

        ys, xs = np.where(poly_mask)
        if len(ys) == 0:
            continue

        feat_samples = feature_stack[ys, xs, :]       # (n_px, F)
        # Drop rows with any NaN
        valid_rows   = ~np.isnan(feat_samples).any(axis=1)
        feat_samples = feat_samples[valid_rows]
        labels       = np.full(valid_rows.sum(), label, dtype=np.int8)

        X_list.append(feat_samples)
        y_list.append(labels)

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    print(f"   Total pixels extracted: {len(y):,}")
    print(f"   Class 0 (Not Paddy): {(y==0).sum():,} | "
          f"Class 1 (Paddy): {(y==1).sum():,}")

    return X, y, feature_names


# ═══════════════════════════════════════════════════════════════════════════════
#  9. SMOTE BALANCING
# ═══════════════════════════════════════════════════════════════════════════════

def balance_samples(X, y, random_state=42):
    print("\n⚖️  Applying SMOTE oversampling …")
    smote = SMOTE(random_state=random_state, k_neighbors=5)
    X_res, y_res = smote.fit_resample(X, y)
    print(f"   After SMOTE → {len(y_res):,} samples "
          f"(0: {(y_res==0).sum()}, 1: {(y_res==1).sum()})")
    return X_res, y_res


# ═══════════════════════════════════════════════════════════════════════════════
#  10. HYPERPARAMETER OPTIMISATION (Optuna)
# ═══════════════════════════════════════════════════════════════════════════════

def optimise_rf(X, y, n_trials, cv_folds, random_state):
    print(f"\n🔧 Optimising Random Forest ({n_trials} trials) …")

    def objective(trial):
        params = {
            "n_estimators"     : trial.suggest_int("n_estimators", 100, 500),
            "max_depth"        : trial.suggest_int("max_depth", 5, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf" : trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features"     : trial.suggest_categorical("max_features",
                                     ["sqrt", "log2", 0.3, 0.5]),
            "class_weight"     : "balanced",
            "random_state"     : random_state,
            "n_jobs"           : -1,
        }
        clf = RandomForestClassifier(**params)
        cv  = StratifiedKFold(n_splits=cv_folds, shuffle=True,
                               random_state=random_state)
        scores = []
        for tr, te in cv.split(X, y):
            clf.fit(X[tr], y[tr])
            scores.append(f1_score(y[te], clf.predict(X[te])))
        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best  = study.best_params
    best.update({"class_weight": "balanced", "random_state": random_state,
                 "n_jobs": -1})
    print(f"   Best RF F1 (CV): {study.best_value:.4f}")
    return RandomForestClassifier(**best)


def optimise_xgb(X, y, n_trials, cv_folds, random_state):
    print(f"\n🔧 Optimising XGBoost ({n_trials} trials) …")
    scale_pos = float((y == 0).sum()) / float((y == 1).sum())

    def objective(trial):
        params = {
            "n_estimators"    : trial.suggest_int("n_estimators", 100, 600),
            "max_depth"       : trial.suggest_int("max_depth", 3, 12),
            "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "gamma"           : trial.suggest_float("gamma", 0, 5),
            "reg_alpha"       : trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
            "reg_lambda"      : trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
            "scale_pos_weight": scale_pos,
            "eval_metric"     : "logloss",
            "random_state"    : random_state,
            "n_jobs"          : -1,
            "use_label_encoder": False,
        }
        clf    = xgb.XGBClassifier(**params)
        cv     = StratifiedKFold(n_splits=cv_folds, shuffle=True,
                                  random_state=random_state)
        scores = []
        for tr, te in cv.split(X, y):
            clf.fit(X[tr], y[tr],
                    eval_set=[(X[te], y[te])],
                    verbose=False)
            scores.append(f1_score(y[te], clf.predict(X[te])))
        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best  = study.best_params
    best.update({
        "scale_pos_weight": scale_pos,
        "eval_metric": "logloss",
        "random_state": random_state,
        "n_jobs": -1,
        "use_label_encoder": False,
    })
    print(f"   Best XGB F1 (CV): {study.best_value:.4f}")
    return xgb.XGBClassifier(**best)


# ═══════════════════════════════════════════════════════════════════════════════
#  11. PROBABILITY THRESHOLD OPTIMISATION
# ═══════════════════════════════════════════════════════════════════════════════

def optimise_threshold(clf, X, y, cv_folds, random_state):
    """Find the probability threshold that maximises F1 on CV predictions."""
    print("   Optimising probability threshold …")
    cv    = StratifiedKFold(n_splits=cv_folds, shuffle=True,
                             random_state=random_state)
    probs = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:, 1]

    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.3, 0.71, 0.01):
        f1 = f1_score(y, (probs >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_t = f1, t

    print(f"   Optimal threshold: {best_t:.2f}  (CV F1={best_f1:.4f})")
    return best_t


# ═══════════════════════════════════════════════════════════════════════════════
#  12. FEATURE IMPORTANCE & SELECTION
# ═══════════════════════════════════════════════════════════════════════════════

def select_features(clf_rf, clf_xgb, feature_names, X, y,
                    top_k: int = None, threshold: float = 0.005):
    """
    Average normalised importances from RF + XGBoost.
    Keep features above threshold (or top-k if specified).
    """
    print("\n📈 Ranking and selecting features …")
    rf_imp  = clf_rf.feature_importances_
    xgb_imp = clf_xgb.feature_importances_

    # Normalise each to sum=1 before averaging
    rf_imp_n  = rf_imp  / (rf_imp.sum()  + 1e-12)
    xgb_imp_n = xgb_imp / (xgb_imp.sum() + 1e-12)
    avg_imp   = (rf_imp_n + xgb_imp_n) / 2.0

    sorted_idx = np.argsort(avg_imp)[::-1]

    if top_k is not None:
        keep_idx = sorted_idx[:top_k]
    else:
        keep_idx = sorted_idx[avg_imp[sorted_idx] >= threshold]

    keep_names = [feature_names[i] for i in keep_idx]
    print(f"   Features retained: {len(keep_names)} / {len(feature_names)}")

    return keep_idx, keep_names, avg_imp, sorted_idx


# ═══════════════════════════════════════════════════════════════════════════════
#  13. CLASSIFY FULL IMAGE (block-wise)
# ═══════════════════════════════════════════════════════════════════════════════

def classify_image(clf, feature_stack, keep_idx, threshold, nodata_mask,
                   block_size=1024):
    """
    Classify the full raster in horizontal blocks to manage memory.

    Returns
    -------
    class_map : (H,W) uint8   — 0=NoData  1=Not Paddy  2=Paddy
    prob_map  : (H,W) float32 — Paddy probability  NaN outside valid mask
    """
    print("\n🗺  Classifying full image (block processing) …")
    H, W, _ = feature_stack.shape
    class_out = np.zeros((H, W), dtype=np.uint8)
    prob_out  = np.full((H, W), np.nan, dtype=np.float32)

    n_blocks = (H + block_size - 1) // block_size

    for b_i in range(n_blocks):
        r0 = b_i * block_size
        r1 = min(r0 + block_size, H)

        block_mask  = nodata_mask[r0:r1, :]
        block_feat  = feature_stack[r0:r1, :, :][:, :, keep_idx]

        valid_px    = block_mask & ~np.isnan(block_feat).any(axis=-1)
        ys, xs      = np.where(valid_px)

        if len(ys) == 0:
            continue

        X_block = block_feat[ys, xs, :]
        probs   = clf.predict_proba(X_block)[:, 1]
        preds   = (probs >= threshold).astype(np.uint8) + 1   # 1=NotPaddy 2=Paddy

        class_out[r0 + ys, xs] = preds
        prob_out[r0  + ys, xs] = probs.astype(np.float32)

        print(f"   Block {b_i+1}/{n_blocks} done  "
              f"(rows {r0}–{r1}, valid={len(ys):,})")

    return class_out, prob_out


# ═══════════════════════════════════════════════════════════════════════════════
#  14. EXPORT RASTERS
# ═══════════════════════════════════════════════════════════════════════════════

def export_raster(array: np.ndarray, meta: dict, path: str,
                  dtype="uint8", nodata=0, compress="lzw"):
    out_meta = meta.copy()
    out_meta.update({
        "count"   : 1,
        "dtype"   : dtype,
        "nodata"  : nodata,
        "compress": compress,
        "driver"  : "GTiff",
    })
    with rasterio.open(path, "w", **out_meta) as dst:
        dst.write(array.astype(dtype), 1)
    print(f"   ✅ Saved → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  15. PLOTS
# ═══════════════════════════════════════════════════════════════════════════════

def plot_feature_importance(avg_imp, sorted_idx, feature_names,
                            top_n=30, save_path=None):
    top_idx   = sorted_idx[:top_n]
    top_names = [feature_names[i] for i in top_idx]
    top_imp   = avg_imp[top_idx]

    fig, ax = plt.subplots(figsize=(10, 8))
    colors  = ["#2ecc71" if "GLCM" in n else
               "#3498db" if any(x in n for x in ["NDVI","EVI","NDWI","SAVI","RVI","DVI","GNDVI","IPVI","OSAVI","MSAVI"]) else
               "#e74c3c" if "Sobel" in n or "Laplace" in n or "Morph" in n else
               "#9b59b6" if "PCA" in n else
               "#f39c12" if "Loc" in n else
               "#1abc9c" if "Ratio" in n else
               "#95a5a6" for n in top_names]

    bars = ax.barh(range(top_n), top_imp[::-1], color=colors[::-1])
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_names[::-1], fontsize=8)
    ax.set_xlabel("Average Normalised Importance (RF + XGBoost)")
    ax.set_title(f"Top-{top_n} Feature Importances — Paddy Classification")

    # Legend
    from matplotlib.patches import Patch
    legend_items = [
        Patch(color="#2ecc71", label="GLCM Texture"),
        Patch(color="#3498db", label="Vegetation Index"),
        Patch(color="#e74c3c", label="Edge/Structure"),
        Patch(color="#9b59b6", label="PCA"),
        Patch(color="#f39c12", label="Local Stats"),
        Patch(color="#1abc9c", label="Band Ratio"),
        Patch(color="#95a5a6", label="Spectral Band"),
    ]
    ax.legend(handles=legend_items, loc="lower right", fontsize=8)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"   ✅ Feature importance plot → {save_path}")
    plt.close()


def plot_confusion_matrix(y_true, y_pred, save_path=None):
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Not Paddy", "Paddy"])

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title("Confusion Matrix — Final Ensemble Model")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"   ✅ Confusion matrix → {save_path}")
    plt.close()


def plot_classified_map(class_map, prob_map, save_path=None):
    """Quick visual of classification + probability side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    cmap_cls = ListedColormap(["#d3d3d3", "#228B22", "#FFD700"])
    axes[0].imshow(class_map, cmap=cmap_cls,
                   vmin=0, vmax=2, interpolation="none")
    axes[0].set_title("Classified Map\n(0=NoData  1=Not Paddy  2=Paddy)")
    axes[0].axis("off")

    im = axes[1].imshow(prob_map, cmap="RdYlGn",
                         vmin=0, vmax=1, interpolation="none")
    axes[1].set_title("Paddy Probability Map")
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], fraction=0.04, pad=0.02)

    plt.suptitle("Borigumma Rabi 2025-26 — Paddy Classification", fontsize=13)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"   ✅ Classification map preview → {save_path}")
    plt.close()


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    t0 = time.time()
    print("=" * 72)
    print(" GeoAI Paddy Classification Pipeline — Borigumma Rabi 2025-26")
    print("=" * 72)

    # ── 1. Read raster ────────────────────────────────────────────────────────
    bands, meta, nodata_mask = read_raster(
        CFG["raster_path"],
        scale=CFG["scale_factor"],
        nodata=CFG["nodata_val"]
    )
    B, G, R, N = bands["Blue"], bands["Green"], bands["Red"], bands["NIR"]

    # ── 2. Vegetation indices ─────────────────────────────────────────────────
    indices = compute_indices(B, G, R, N)
    # Make NDVI accessible for local stats
    bands["NDVI"] = indices["NDVI"]

    # ── 3. GLCM texture ───────────────────────────────────────────────────────
    # Use distance=1 only and 2 angles for tractable runtime on large images
    glcm_feats = compute_all_glcm(
        bands,
        win=9,
        distances=[1],
        angles=[0, np.pi/2]
    )

    # ── 4. Local spatial statistics ───────────────────────────────────────────
    local_st = compute_local_stats(bands, windows=CFG["window_sizes"])

    # ── 5. Edge features ──────────────────────────────────────────────────────
    edges = compute_edge_features(bands)

    # ── 6. PCA ────────────────────────────────────────────────────────────────
    pca_feats = compute_pca(
        bands, n_components=CFG["n_pca_components"],
        nodata_mask=nodata_mask
    )

    # ── 7. Band ratios + entropy ──────────────────────────────────────────────
    ratios       = compute_band_ratios(bands)
    entropy_feats= compute_entropy_filter(bands, radius=3)

    # ── 8. Build feature stack ────────────────────────────────────────────────
    feature_stack, feature_names = build_feature_stack(
        bands, indices, glcm_feats, local_st, edges,
        pca_feats, ratios, entropy_feats
    )

    # ── 9. Extract training samples ───────────────────────────────────────────
    X, y, feat_names = extract_samples(
        CFG["shp_path"], CFG["raster_path"],
        feature_stack, feature_names,
        CFG["class_field"], meta
    )

    # ── 10. Balance ───────────────────────────────────────────────────────────
    X_bal, y_bal = balance_samples(X, y, CFG["random_state"])

    # Scale for XGBoost (RF doesn't need it but doesn't hurt)
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_bal)

    # ── 11. Optimise RF ───────────────────────────────────────────────────────
    clf_rf = optimise_rf(
        X_bal, y_bal,
        CFG["n_optuna_trials"], CFG["cv_folds"], CFG["random_state"]
    )
    clf_rf.fit(X_bal, y_bal)

    # ── 12. Optimise XGBoost ──────────────────────────────────────────────────
    clf_xgb = optimise_xgb(
        X_scaled, y_bal,
        CFG["n_optuna_trials"], CFG["cv_folds"], CFG["random_state"]
    )
    clf_xgb.fit(X_scaled, y_bal, verbose=False)

    # ── 13. Feature selection ─────────────────────────────────────────────────
    keep_idx, keep_names, avg_imp, sorted_idx = select_features(
        clf_rf, clf_xgb, feat_names, X_bal, y_bal,
        threshold=0.003
    )

    # Re-train final RF on selected features only
    print("\n🏋️  Re-training final Random Forest on selected features …")
    X_sel = X_bal[:, keep_idx]
    clf_final = RandomForestClassifier(
        **{k: v for k, v in clf_rf.get_params().items()},
    )
    clf_final.fit(X_sel, y_bal)

    # ── 14. Probability threshold optimisation ────────────────────────────────
    threshold = optimise_threshold(
        clf_final, X_sel, y_bal, CFG["cv_folds"], CFG["random_state"]
    )

    # ── 15. Final accuracy metrics ────────────────────────────────────────────
    print("\n📋 Final Cross-Validated Evaluation Metrics:")
    cv_final = StratifiedKFold(n_splits=CFG["cv_folds"], shuffle=True,
                                random_state=CFG["random_state"])
    y_pred_cv = cross_val_predict(clf_final, X_sel, y_bal, cv=cv_final)
    print(classification_report(y_bal, y_pred_cv,
                                  target_names=["Not Paddy", "Paddy"],
                                  digits=4))

    roc = roc_auc_score(
        y_bal,
        cross_val_predict(clf_final, X_sel, y_bal, cv=cv_final,
                           method="predict_proba")[:, 1]
    )
    print(f"   ROC-AUC: {roc:.4f}")

    # ── 16. Classify full image ───────────────────────────────────────────────
    class_map, prob_map = classify_image(
        clf_final, feature_stack, keep_idx, threshold,
        nodata_mask, block_size=512
    )

    # ── 17. Export rasters ────────────────────────────────────────────────────
    print("\n💾 Exporting output rasters …")
    cls_path  = str(out / "Borigumma_Paddy_Classified.tif")
    prob_path = str(out / "Borigumma_Paddy_Probability.tif")

    export_raster(class_map, meta, cls_path,
                  dtype="uint8", nodata=0, compress="lzw")

    # Probability: scale to 0–100 uint8 to keep file small
    prob_uint8 = np.where(np.isnan(prob_map), 255,
                           (prob_map * 100).astype(np.uint8))
    export_raster(prob_uint8, meta, prob_path,
                  dtype="uint8", nodata=255, compress="lzw")

    # ── 18. Plots ─────────────────────────────────────────────────────────────
    print("\n📊 Generating plots …")
    plot_feature_importance(
        avg_imp, sorted_idx, feat_names,
        top_n=30,
        save_path=str(out / "Feature_Importance.png")
    )
    plot_confusion_matrix(
        y_bal, y_pred_cv,
        save_path=str(out / "Confusion_Matrix.png")
    )
    plot_classified_map(
        class_map, prob_map,
        save_path=str(out / "Classification_Preview.png")
    )

    elapsed = (time.time() - t0) / 60
    print(f"\n{'='*72}")
    print(f" ✅ Pipeline complete in {elapsed:.1f} minutes")
    print(f" Output directory: {out}")
    print(f"{'='*72}")

    # ── Summary ───────────────────────────────────────────────────────────────
    paddy_px     = int((class_map == 2).sum())
    not_paddy_px = int((class_map == 1).sum())
    px_area_ha   = (3.0 * 3.0) / 10000.0  # 3m × 3m → ha

    print(f"\n🌾 Paddy area     : {paddy_px * px_area_ha:,.1f} ha")
    print(f"   Not-Paddy area : {not_paddy_px * px_area_ha:,.1f} ha")
    print(f"   NoData pixels  : {int((class_map == 0).sum()):,}")


# ─── Entry point ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


✅ All imports successful
📁 Output directory: E:\Rabi_Paddy_2526\LOT1\Koraput\Output
 GeoAI Paddy Classification Pipeline — Borigumma Rabi 2025-26

📡 Reading raster …
   Shape: 11689 × 15489  |  Valid pixels: 112,015,018
🌿 Computing vegetation indices …
   Generated 10 indices

🔲 Computing GLCM texture features …
   GLCM [NIR] 11689×15489 win=9  … (may take a minute)


MemoryError: Unable to allocate 1.00 MiB for an array with shape (256, 256, 1, 2) and data type float64